# Bodega UL — Análisis de inventario con Pandas y Seaborn

Temas del curso aplicados aquí: **3.18 Pandas at Work**, **3.19 Data Diving**, **3.20/3.21 Visualización con Seaborn**, **3.23 Intuición estadística**.

Este notebook se conecta de forma **solo lectura** a la misma base de datos de Supabase que usa la web app (usa la `anon key`, protegida por las políticas de RLS que solo permiten `SELECT`), jala los datos con `pandas` y genera gráficas con `seaborn` para dárselas a Student Life en su junta mensual.

In [ ]:
!pip install -q supabase pandas seaborn matplotlib

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from supabase import create_client

# Reemplaza con tus propios valores (los mismos que usaste en Vercel).
# La anon key es de solo-lectura para estas tablas: no puede borrar ni modificar nada.
SUPABASE_URL = "https://uxrhxinzchtuvsuvwjdp.supabase.co"
SUPABASE_ANON_KEY = "PEGA_AQUI_TU_SUPABASE_ANON_KEY"

supabase = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)

sns.set_theme(style="darkgrid")

## 1. Cargar los datos (pandas)

In [ ]:
articulos = pd.DataFrame(supabase.table("articulos").select("*").execute().data)
prestamos = pd.DataFrame(supabase.table("prestamos").select("*").execute().data)
devoluciones = pd.DataFrame(supabase.table("devoluciones").select("*").execute().data)

print(f"Artículos: {len(articulos)} | Préstamos: {len(prestamos)} | Devoluciones: {len(devoluciones)}")
articulos.head()

## 2. Limpieza rápida (Tema 3.22 — Data Cleaning)

In [ ]:
for df in (articulos, prestamos, devoluciones):
    if not df.empty:
        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].astype(str).str.strip()

if not prestamos.empty:
    prestamos["fecha_prestamo"] = pd.to_datetime(prestamos["fecha_prestamo"])
if not devoluciones.empty:
    devoluciones["fecha_devolucion"] = pd.to_datetime(devoluciones["fecha_devolucion"])

## 3. Artículos por club (bar chart)

In [ ]:
plt.figure(figsize=(8, 5))
orden = articulos["club"].value_counts().index
sns.countplot(data=articulos, y="club", order=orden, palette="flare")
plt.title("Artículos registrados por club")
plt.xlabel("Cantidad de artículos")
plt.ylabel("Club")
plt.tight_layout()
plt.show()

## 4. Artículos más prestados (top 10)

In [ ]:
if not prestamos.empty:
    conteo = prestamos.merge(articulos[["sku", "nombre_articulo"]], on="sku", how="left")
    top10 = conteo["nombre_articulo"].value_counts().head(10)

    plt.figure(figsize=(8, 5))
    sns.barplot(x=top10.values, y=top10.index, palette="crest")
    plt.title("Top 10 artículos más prestados")
    plt.xlabel("Veces prestado")
    plt.tight_layout()
    plt.show()
else:
    print("Todavía no hay préstamos registrados.")

## 5. Estado de calidad al regresar, por club (heatmap)

In [ ]:
if not devoluciones.empty:
    dev_con_club = devoluciones.merge(articulos[["sku", "club"]], on="sku", how="left")
    tabla = pd.crosstab(dev_con_club["club"], dev_con_club["estado_calidad_regreso"])

    plt.figure(figsize=(8, 5))
    sns.heatmap(tabla, annot=True, fmt="d", cmap="rocket_r")
    plt.title("Estado de calidad al regresar, por club")
    plt.tight_layout()
    plt.show()

    tasa_perdida = (dev_con_club["estado_calidad_regreso"] == "perdido").groupby(dev_con_club["club"]).mean().sort_values(ascending=False)
    print("Tasa de pérdida por club (Tema 3.23 — intuición estadística):")
    print((tasa_perdida * 100).round(1).astype(str) + "%")
else:
    print("Todavía no hay devoluciones registradas.")

## 6. Duración promedio de préstamo (días) por club

In [ ]:
if not devoluciones.empty and not prestamos.empty:
    cruce = devoluciones.merge(
        prestamos[["id", "sku", "fecha_prestamo"]].rename(columns={"id": "prestamo_id"}),
        on="prestamo_id", how="left"
    ).merge(articulos[["sku", "club"]], on="sku", how="left")

    cruce["dias_prestado"] = (cruce["fecha_devolucion"] - cruce["fecha_prestamo"]).dt.days

    plt.figure(figsize=(8, 5))
    sns.boxplot(data=cruce, x="dias_prestado", y="club", palette="mako")
    plt.title("Distribución de días de préstamo por club")
    plt.tight_layout()
    plt.show()

    print("Promedio de días prestado, por club:")
    print(cruce.groupby("club")["dias_prestado"].mean().round(1))
else:
    print("Faltan préstamos o devoluciones para calcular duración.")

## Notas
- Este notebook es **solo lectura**: la `anon key` no puede insertar/borrar en estas tablas más que lo que las políticas de RLS permiten, y aquí solo se usa `.select()`.
- Corre este notebook una vez a la semana (o antes de cada junta de Student Life) para tener las gráficas actualizadas.
- Si alguna gráfica sale vacía, es porque todavía no hay suficientes registros de ese tipo — vuelve a correrlo cuando la bodega lleve más movimiento.